# Лабораторная работа №1
## Пайплайны обработки данных на основе LangChain и LLM

Notebook запускается после настройки `.env`.

In [ ]:
from src.lab01 import *
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
init_database()
print('Каталог создан')

## Часть 1. Базовая LCEL-цепочка
Следующая ячейка демонстрирует `AIMessage` при прямом вызове и строку после `StrOutputParser`, а также `batch`.

In [ ]:
llm = create_llm(enable_thinking=False)
direct = llm.invoke('Определи категорию: хочу планшет')
chain = basic_chain(llm)
batch_result = chain.batch([
    {'query': 'Нужен ноутбук', 'language': 'русском'},
    {'query': 'Ищу смартфон', 'language': 'русском'},
    {'query': 'Хочу планшет', 'language': 'русском'},
])
print(type(direct).__name__, direct.content)
print(type(batch_result[0]).__name__, batch_result)

## Часть 2. Structured Output и Pydantic
`ProductQuery` ограничивает значения категории, цены и сортировки. Инструкции парсера добавляются в system prompt.

In [ ]:
structured, parser = structured_chain(llm)
tests = [
    'Найди ноутбуки дешевле 70 000 рублей',
    'Покажи самые дорогие смартфоны',
    'Хочу посмотреть планшеты',
    'Покажи ноутбуки до 50 тысяч',
    'Найди холодильники',
    'Покажи недорогие ноутбуки',
    'А сначала дорогие?'
]
for text in tests:
    result = structured.invoke({'query': text})
    print(text, '->', result.model_dump_json())

## Часть 3. SQLite и инструмент
Поиск получает только `ProductQuery` и строит SQL с параметрами. При `other` поиск не выполняется.

In [ ]:
query = ProductQuery(category='laptop', max_price=70000, sort_by='price_asc')
result = search_products(query)
result

# search_tool = build_search_tool()
# response = llm.bind_tools([search_tool]).invoke('Найди ноутбуки до 70 тысяч')
# print(response.content, response.tool_calls)
# print(handle_tool_calls(response))
# print(handle_tool_calls(AIMessage(content='Инструмент не вызван')))

## Часть 4. Retry и журнал
FakeMessagesListChatModel делает эксперимент воспроизводимым: первая попытка возвращает невалидный JSON, вторая — валидный.

In [ ]:
fake_llm = FakeMessagesListChatModel(responses=[
    AIMessage(content='некорректный JSON'),
    AIMessage(content='{\"category\": \"laptop\", \"max_price\": 70000, \"sort_by\": \"price_asc\"}')
])
without_retry, _ = structured_chain(fake_llm)
try:
    without_retry.invoke({'query': 'ноутбуки'})
except Exception as error:
    print('Без retry:', type(error).__name__)

fake_llm = FakeMessagesListChatModel(responses=[
    AIMessage(content='некорректный JSON'),
    AIMessage(content='{\"category\": \"laptop\", \"max_price\": 70000, \"sort_by\": \"price_asc\"}')
])
retry_result, retry_log = run_logged_search(fake_llm, {'query': 'ноутбуки'})
print(retry_result.message)
print(retry_log.as_json())

invalid_examples = ['{category: laptop}', '{\"category\":\"camera\"}', '{\"category\":\"laptop\",\"max_price\":-5}']
for text in invalid_examples:
    try:
        structured_chain(fake_llm)[1].parse(text)
    except Exception as error:
        print(type(error).__name__, error)

## Часть 5. Состояние диалога
В интерактивном режиме последняя валидная модель хранится в `Conversation`, а не внутри LLM. Полный сравнительный прогон результатов запускается отдельным скриптом.

In [ ]:
# python -m scripts.run_experiments
# Результаты сохраняются в results/experiment_results.json.
!python -m unittest discover -s tests -v